# Demo Text-to-SQL con Snowflake Cortex Analyst

**Master AI – Clases IA**

En este notebook construirás, paso a paso, una demo de **text-to-SQL**: desde
los ficheros CSV originales hasta una app de Streamlit que responde preguntas
en lenguaje natural usando Cortex Analyst.

La idea: cargas unos datos de ventas en Snowflake, los describes con una
**Semantic View** y dejas que Cortex Analyst traduzca preguntas como
*"compara ingresos reales vs previstos por mes"* en SQL, ejecute la consulta y
muestre el resultado.

Trabajarás sobre **Snowsight** (la interfaz web de Snowflake) para los pasos de
carga y de creación de la Semantic View, y sobre **Snowflake Notebooks /
Streamlit in Snowflake** para el resto.

## Índice
1. Punto de partida: los datos
2. Crear los objetos en Snowflake (BD, esquema, warehouse, stage, tablas)
3. Subir los CSV al stage
4. Cargar los datos en las tablas (COPY INTO)
5. Revisar los datos cargados
6. Crear la Semantic View para Cortex Analyst
7. Probar Cortex Analyst desde SQL
8. La app de Streamlit (text-to-SQL end-to-end)
9. Resumen del flujo completo


## 1. Punto de partida: los datos

Partimos de tres ficheros CSV de un dataset de **ingresos diarios** (serie
temporal de ventas):

| Fichero | Tipo | Tabla destino |
|---|---|---|
| `daily_revenue.csv` | Tabla de hechos (*fact*) | `daily_revenue` |
| `product.csv` | Dimensión | `product_dim` |
| `region.csv` | Dimensión | `region_dim` |

**Columnas de cada CSV:**

- `daily_revenue.csv` → `DATE, REVENUE, COGS, FORECASTED_REVENUE, Product_id, Region_id`
- `product.csv` → `Product_id, Product_line`
- `region.csv` → `Region_id, Region, State`

El modelo es un **esquema en estrella**: una tabla de hechos (`daily_revenue`)
que se une a dos dimensiones (`product_dim`, `region_dim`) mediante `Product_id`
y `Region_id`.

> Todos los CSV usan **coma** `,` como separador y traen cabecera en la
> primera fila.


In [ ]:
# Inspecciona los CSV antes de subirlos a Snowflake.
import pandas as pd

daily = pd.read_csv("cortex_data/daily_revenue.csv")
product = pd.read_csv("cortex_data/product.csv")
region = pd.read_csv("cortex_data/region.csv")

print("daily_revenue:", daily.shape)
print(daily.head(3), "\n")
print("product:", product.shape)
print(product, "\n")
print("region:", region.shape)
print(region.head(5))

## 2. Crear los objetos en Snowflake

Crea la base de datos `cortex_analyst_demo`, el esquema `revenue_timeseries`,
un warehouse, un rol con permisos de Cortex, un stage para los CSV y las tres
tablas.

Abre un **worksheet de Snowsight** (Projects » Worksheets), pega el siguiente
SQL y ejecútalo con *Run All*. Sustituye `<user>` por tu usuario de Snowflake.


In [ ]:
/*--
  Rol, base de datos, esquema, warehouse y stage
--*/
USE ROLE SECURITYADMIN;

CREATE ROLE IF NOT EXISTS cortex_user_role;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE cortex_user_role;
GRANT ROLE cortex_user_role TO USER <user>;

USE ROLE sysadmin;

CREATE OR REPLACE DATABASE cortex_analyst_demo;
CREATE OR REPLACE SCHEMA cortex_analyst_demo.revenue_timeseries;

CREATE OR REPLACE WAREHOUSE cortex_analyst_wh
    WAREHOUSE_SIZE = 'large'
    WAREHOUSE_TYPE = 'standard'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Warehouse for Cortex Analyst demo';

GRANT USAGE   ON WAREHOUSE cortex_analyst_wh TO ROLE cortex_user_role;
GRANT OPERATE ON WAREHOUSE cortex_analyst_wh TO ROLE cortex_user_role;
GRANT OWNERSHIP ON SCHEMA   cortex_analyst_demo.revenue_timeseries TO ROLE cortex_user_role;
GRANT OWNERSHIP ON DATABASE cortex_analyst_demo TO ROLE cortex_user_role;

USE ROLE cortex_user_role;
USE WAREHOUSE cortex_analyst_wh;
USE DATABASE cortex_analyst_demo;
USE SCHEMA cortex_analyst_demo.revenue_timeseries;

-- Stage para los CSV en crudo
CREATE OR REPLACE STAGE raw_data DIRECTORY = (ENABLE = TRUE);

/*--
  Tablas: 1 de hechos + 2 dimensiones
--*/
CREATE OR REPLACE TABLE cortex_analyst_demo.revenue_timeseries.daily_revenue (
    date               DATE,
    revenue            FLOAT,
    cogs               FLOAT,
    forecasted_revenue FLOAT,
    product_id         INT,
    region_id          INT
);

CREATE OR REPLACE TABLE cortex_analyst_demo.revenue_timeseries.product_dim (
    product_id   INT,
    product_line VARCHAR(16777216)
);

CREATE OR REPLACE TABLE cortex_analyst_demo.revenue_timeseries.region_dim (
    region_id    INT,
    sales_region VARCHAR(16777216),
    state        VARCHAR(16777216)
);

## 3. Subir los CSV al stage

Sube los CSV al stage `raw_data` desde Snowsight:

1. Menú lateral → **Ingestion** → **Add Data** → **Load files into a stage**.
2. Arrastra los tres CSV: `daily_revenue.csv`, `product.csv`, `region.csv`.
3. Elige la base de datos `cortex_analyst_demo` y el stage `raw_data`.
4. Pulsa **Upload**.


## 4. Cargar los datos en las tablas (COPY INTO)

Con los CSV ya en el stage, cópialos a las tablas con `COPY INTO`. Ejecuta el
siguiente SQL en un worksheet de Snowsight:


In [ ]:
USE WAREHOUSE cortex_analyst_wh;

COPY INTO cortex_analyst_demo.revenue_timeseries.daily_revenue
FROM @raw_data
FILES = ('daily_revenue.csv')
FILE_FORMAT = (
    TYPE = CSV,
    SKIP_HEADER = 1,
    FIELD_DELIMITER = ',',
    TRIM_SPACE = FALSE,
    FIELD_OPTIONALLY_ENCLOSED_BY = NONE,
    REPLACE_INVALID_CHARACTERS = TRUE,
    DATE_FORMAT = AUTO, TIME_FORMAT = AUTO, TIMESTAMP_FORMAT = AUTO,
    EMPTY_FIELD_AS_NULL = FALSE,
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR = CONTINUE
FORCE = TRUE;

COPY INTO cortex_analyst_demo.revenue_timeseries.product_dim
FROM @raw_data
FILES = ('product.csv')
FILE_FORMAT = (
    TYPE = CSV, SKIP_HEADER = 1, FIELD_DELIMITER = ',',
    TRIM_SPACE = FALSE, FIELD_OPTIONALLY_ENCLOSED_BY = NONE,
    REPLACE_INVALID_CHARACTERS = TRUE,
    DATE_FORMAT = AUTO, TIME_FORMAT = AUTO, TIMESTAMP_FORMAT = AUTO,
    EMPTY_FIELD_AS_NULL = FALSE, ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR = CONTINUE
FORCE = TRUE;

COPY INTO cortex_analyst_demo.revenue_timeseries.region_dim
FROM @raw_data
FILES = ('region.csv')
FILE_FORMAT = (
    TYPE = CSV, SKIP_HEADER = 1, FIELD_DELIMITER = ',',
    TRIM_SPACE = FALSE, FIELD_OPTIONALLY_ENCLOSED_BY = NONE,
    REPLACE_INVALID_CHARACTERS = TRUE,
    DATE_FORMAT = AUTO, TIME_FORMAT = AUTO, TIMESTAMP_FORMAT = AUTO,
    EMPTY_FIELD_AS_NULL = FALSE, ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
)
ON_ERROR = CONTINUE
FORCE = TRUE;

## 5. Revisar los datos cargados

Comprueba que las tres tablas tienen datos. Ejecuta estas consultas en un
worksheet (o en una celda SQL de un Snowflake Notebook):


In [ ]:
SELECT COUNT(*) AS filas_daily   FROM cortex_analyst_demo.revenue_timeseries.daily_revenue;
SELECT * FROM cortex_analyst_demo.revenue_timeseries.daily_revenue LIMIT 5;

SELECT * FROM cortex_analyst_demo.revenue_timeseries.product_dim;
SELECT * FROM cortex_analyst_demo.revenue_timeseries.region_dim LIMIT 5;

## 6. Crear la Semantic View para Cortex Analyst

La **Semantic View** (o *semantic model*) es la capa que permite a Cortex
Analyst entender el significado de las tablas: qué es una dimensión, qué es una
métrica, cómo se relacionan las tablas y qué sinónimos en lenguaje natural usar.

Créala desde Snowsight: → **AI & ML** → **Cortex Analyst** → crear una nueva
Semantic View. Define:

- **Tablas lógicas:** `daily_revenue` (hechos), `product_dim`, `region_dim`.
- **Relaciones (joins):** `daily_revenue.product_id = product_dim.product_id`
  y `daily_revenue.region_id = region_dim.region_id`.
- **Dimensiones:** `date`, `product_line`, `sales_region`, `state`.
- **Métricas / facts:** `revenue`, `cogs`, `forecasted_revenue`.
- **Sinónimos:** p. ej. *"ingresos"/"facturación"* → `revenue`,
  *"previsión"* → `forecasted_revenue`.

> 💡 La calidad de las respuestas depende de esta capa: cuanto mejor definas
> dimensiones, métricas, joins y sinónimos, mejor SQL generará Cortex.

### Nombre de la Semantic View

El nombre completo es `<BASE_DE_DATOS>.<ESQUEMA>.<NOMBRE_VISTA>` y depende de
qué base de datos y esquema elijas al crearla. **Apunta el nombre que le
pongas**, porque tendrás que escribirlo en la variable `SEMANTIC_VIEW` de
`app.py` (paso 8). Si ambos no coinciden, la app no encontrará la Semantic View.

Para mantenerlo ordenado, puedes crear la Semantic View en la misma base de
datos y esquema que las tablas (`cortex_analyst_demo.revenue_timeseries`).


## 7. Probar Cortex Analyst desde SQL

Cortex Analyst se consume vía **API REST** (`/api/v2/cortex/analyst/message`).
La respuesta incluye el **SQL generado**, que luego se ejecuta normalmente.
El ciclo es:

1. Pregunta en lenguaje natural →
2. Cortex Analyst (con la Semantic View) genera SQL →
3. Snowflake ejecuta el SQL →
4. Se muestran tabla + gráfico.

Ejemplo del tipo de SQL que Cortex genera por debajo para
*"compara ingresos reales vs previstos por mes"*:


In [ ]:
SELECT
    DATE_TRUNC('month', date) AS mes,
    SUM(revenue)              AS ingresos_reales,
    SUM(forecasted_revenue)  AS ingresos_previstos
FROM cortex_analyst_demo.revenue_timeseries.daily_revenue
GROUP BY 1
ORDER BY 1;

## 8. La app de Streamlit (text-to-SQL end-to-end)

La pieza final es una app de **Streamlit in Snowflake** (`app.py`). El usuario
escribe una pregunta, la app llama a Cortex Analyst por REST, ejecuta el SQL
devuelto y pinta el resultado. Al ejecutarse dentro de Snowflake usa
`get_active_session()`, así que no hace falta poner usuario ni contraseña en el
código.

Qué hace cada función clave:

- `get_active_session()` → usa la sesión de Snowflake ya autenticada.
- `get_session_token()` → obtiene el token para llamar a la API REST.
- `ask_cortex(question)` → envía la pregunta + la Semantic View a Cortex Analyst.
- `run_sql(sql)` → ejecuta el SQL devuelto y lo trae como DataFrame.
- `show_cortex_response(...)` → muestra texto, SQL, tabla y gráfico.

Extracto de la lógica central (el código completo está en `app.py`):


```python
import pandas as pd
import requests
import streamlit as st
from snowflake.snowpark.context import get_active_session

SNOWFLAKE_HOST = "hb18782.eu-west-3.aws.snowflakecomputing.com"
SEMANTIC_VIEW = "DEMO.PUBLIC.DEMO_EBIS"
CORTEX_ANALYST_ENDPOINT = "/api/v2/cortex/analyst/message"

session = get_active_session()


def get_session_token() -> str:
    conn = session.connection
    rest = getattr(conn, "rest", None)
    if rest is not None and hasattr(rest, "token"):
        return rest.token
    inner_conn = getattr(conn, "_conn", None)
    if inner_conn is not None:
        inner_rest = getattr(inner_conn, "rest", None)
        if inner_rest is not None and hasattr(inner_rest, "token"):
            return inner_rest.token
    raise RuntimeError("No se pudo obtener el token de sesión de Snowflake.")


def ask_cortex(question: str) -> dict:
    token = get_session_token()
    url = f"https://{SNOWFLAKE_HOST}{CORTEX_ANALYST_ENDPOINT}"
    request_body = {
        "messages": [
            {"role": "user", "content": [{"type": "text", "text": question}]}
        ],
        "semantic_view": SEMANTIC_VIEW,
    }
    headers = {
        "Authorization": f'Snowflake Token="{token}"',
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, json=request_body, timeout=60)
    if response.status_code >= 400:
        raise RuntimeError(response.text)
    result = response.json()
    result["request_id"] = response.headers.get("X-Snowflake-Request-Id")
    return result


def run_sql(sql: str) -> pd.DataFrame:
    return session.sql(sql).to_pandas()
```

> Cambia `SNOWFLAKE_HOST` por el host de tu cuenta y `SEMANTIC_VIEW` por el
> nombre de la Semantic View que creaste en el paso 6. El `app.py` completo
> incluye además las funciones de visualización y la interfaz (sidebar,
> botones de ejemplo, etc.).


### Cómo desplegar la app en Snowflake

1. Snowsight → **Projects** → **Streamlit** → **+ Streamlit App**.
2. Elige base de datos/esquema y un warehouse (`cortex_analyst_wh`).
3. Pega el contenido de `app.py`.
4. Pulsa **Run** y prueba con las preguntas de ejemplo de la barra lateral.


## 9. Resumen del flujo completo

```
3 CSV (daily_revenue, product, region)
        │  (subida al stage @raw_data vía Snowsight)
        ▼
Stage raw_data
        │  (COPY INTO)
        ▼
Tablas en cortex_analyst_demo.revenue_timeseries
   (daily_revenue + product_dim + region_dim)
        │  (Semantic View vía Cortex Analyst UI)
        ▼
Semantic View  (capa semántica: dimensiones, métricas, joins, sinónimos)
        │
        ▼
App Streamlit ──► Cortex Analyst (REST) ──► genera SQL
        ▲                                        │
        │                                        ▼
   tabla + gráfico ◄──── Snowflake ejecuta el SQL
```

**En una frase:** subimos 3 CSV a Snowflake (stage → tablas), los describimos
con una Semantic View, y Cortex Analyst traduce preguntas en lenguaje natural
a SQL que Streamlit muestra como tabla y gráfico.
